### Import librairies

In [71]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from scipy.optimize import brentq

# Find the project root so this notebook also works when launched from a subfolder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "python_module" / "pricing_model.py").is_file():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find python_module/pricing_model.py from the notebook directory")

pd.set_option("display.float_format", lambda x: f"{x:.4g}")

from python_module.pricing_model import SABRModel

### Custom functions

In [72]:
def solve_delta_strike(target_delta, option_type):
    """Strike whose BS delta (on the SABR smile) equals target_delta, e.g. -0.25 for a 25d put."""
    def objective(K):
        return SABRModel.compute_option(F0, K, T, alpha, beta, rho, nu, r, option_type)["delta"] - target_delta
    return brentq(objective, F0 * 0.2, F0 * 5.0)


def compute_risks(K, option_type):
    """PV, BS greeks and SABR parameter bumps (full reprice) of one option."""
    base = SABRModel.compute_option(F0, K, T, alpha, beta, rho, nu, r, option_type)

    def bumped_pv(d_alpha=0.0, d_rho=0.0, d_nu=0.0):
        return SABRModel.compute_option(
            F0, K, T, alpha + d_alpha, beta, rho + d_rho, nu + d_nu, r, option_type, compute_bs_greeks=False
        )["price"]

    return {
        "strike": K,
        "PV": base["price"],
        "bs_delta": base["delta"],
        "bs_vega": base["vega"],                           # per 1 vol point
        "bs_gamma_cash": base["gamma"] * F0 ** 2 / 100,    # cash gamma for a 1% move
        "bs_theta": base["theta"],                         # per business day
        "alpha_bump": bumped_pv(d_alpha=alpha_bump_size) - base["price"],
        "rho_bump": bumped_pv(d_rho=rho_bump_size) - base["price"],
        "nu_bump": bumped_pv(d_nu=nu_bump_size) - base["price"],
    }

### Inputs

In [73]:
# Market parameters
F0 = 100.0                                  # forward (spot = forward, no carry)
T = 21 / 252                                # maturity: 1 month (years)
r = 0.00                                    # risk-free rate

# SABR parameters (Hagan 2002, lognormal backbone)
alpha = 0.20                                # initial vol level (~ ATM lognormal vol when beta = 1)
beta = 1.0                                  # CEV exponent: 1 = lognormal, 0 = normal
rho = -0.70                                 # spot / vol correlation (equity skew)
nu = 1.00                                   # vol of vol

# Bump sizes (absolute)
alpha_bump_size = 0.01
rho_bump_size = 0.10
nu_bump_size = 0.50

# Option universe parameters
min_delta_strike = -0.01                    # 1d put
max_delta_strike = 0.01                     # 1d call
n_options = 100

### SABR dynamic

In [74]:
# 100 strikes evenly spaced from the 1d put strike to the 1d call strike, OTM options (put below the forward, call above)
K_min = solve_delta_strike(min_delta_strike, "put")
K_max = solve_delta_strike(max_delta_strike, "call")
strikes = np.linspace(K_min, K_max, n_options)

universe = pd.DataFrame(
    [{"option_type": ot, **compute_risks(K, ot)} for K in strikes for ot in ["put" if K < F0 else "call"]]
)
risk_cols = ["PV", "bs_delta", "bs_vega", "bs_gamma_cash", "bs_theta", "alpha_bump", "rho_bump", "nu_bump"]
universe = universe[["strike", "option_type", *risk_cols]]

print(f"1d put strike = {K_min:.2f}, 1d call strike = {K_max:.2f}")
with pd.option_context("display.max_rows", None):
    display(universe)

1d put strike = 84.08, 1d call strike = 111.94


,strike,option_type,PV,bs_delta,bs_vega,bs_gamma_cash,bs_theta,alpha_bump,rho_bump,nu_bump
0,84.08,put,0.02629,-0.01,0.007694,0.3518,-0.004808,0.008452,-0.003547,0.03171
1,84.36,put,0.02892,-0.01094,0.008322,0.3822,-0.005176,0.009116,-0.003796,0.03328
2,84.64,put,0.0318,-0.01197,0.008994,0.415,-0.005569,0.009826,-0.004058,0.0349
3,84.93,put,0.03495,-0.01308,0.009713,0.4503,-0.005987,0.01058,-0.004333,0.03657
4,85.21,put,0.03839,-0.01429,0.01048,0.4882,-0.006431,0.01139,-0.004622,0.03828
5,85.49,put,0.04215,-0.01559,0.0113,0.5289,-0.006903,0.01225,-0.004925,0.04005
6,85.77,put,0.04626,-0.01701,0.01218,0.5725,-0.007403,0.01316,-0.005241,0.04185
7,86.05,put,0.05074,-0.01855,0.01311,0.6192,-0.007934,0.01414,-0.00557,0.0437
8,86.33,put,0.05563,-0.02022,0.01411,0.6693,-0.008495,0.01517,-0.005913,0.04559
9,86.61,put,0.06097,-0.02202,0.01516,0.7227,-0.009088,0.01626,-0.006269,0.04751


In [75]:
# One panel per risk: the scales are too different to share a y axis
fig = make_subplots(rows=2, cols=4, subplot_titles=risk_cols, horizontal_spacing=0.06, vertical_spacing=0.15)
for i, col in enumerate(risk_cols):
    fig.add_trace(
        go.Scatter(x=universe["strike"], y=universe[col], mode="lines", name=col,
                   hovertemplate="K=%{x:.2f}<br>" + col + "=%{y:.4g}<extra></extra>"),
        row=i // 4 + 1, col=i % 4 + 1,
    )
fig.add_vline(x=F0, line_dash="dot", line_color="grey")
fig.update_xaxes(title_text="strike", row=2)
fig.update_layout(title="SABR universe: risks by strike (OTM options, 1M)", height=650, showlegend=False)
fig.show()

### Subportfolio

In [76]:
# 3 instruments: 25d put, ATM, 25d call
legs = {
    "25d put": (solve_delta_strike(-0.25, "put"), "put"),
    "ATM": (F0, "call"),
    "25d call": (solve_delta_strike(0.25, "call"), "call"),
}
legs_risks = pd.DataFrame({name: compute_risks(K, ot) for name, (K, ot) in legs.items()}).T.astype(float)
display(legs_risks)

# Each portfolio has a unit exposure to its own bump and zero exposure to the two others:
# solve  B @ w = e_i  with B[i, j] = bump i of option j  (3 equations, 3 weights)
bump_cols = ["alpha_bump", "rho_bump", "nu_bump"]
B = legs_risks[bump_cols].T.values
weights = pd.DataFrame(
    np.linalg.solve(B, np.eye(3)), index=legs_risks.index, columns=["alpha_portfolio", "rho_portfolio", "nu_portfolio"]
)
display(weights)

# Portfolio risks: the bump block is the identity, BS greeks are whatever is left over
portfolio_risks = weights.T @ legs_risks[risk_cols]
display(portfolio_risks)

,strike,PV,bs_delta,bs_vega,bs_gamma_cash,bs_theta,alpha_bump,rho_bump,nu_bump
25d put,96.1,0.9505,-0.25,0.09173,5.145,-0.04673,0.09209,-0.01257,0.06829
ATM,100,2.3,0.5115,0.1151,6.914,-0.05476,0.1146,0.004077,0.001939
25d call,103.9,0.7837,0.25,0.09173,5.888,-0.04083,0.09229,0.02273,-0.05306


,alpha_portfolio,rho_portfolio,nu_portfolio
25d put,-5.25,126.2,44.96
ATM,17.85,-225.6,-65.59
25d call,-6.104,154.2,36.61


,PV,bs_delta,bs_vega,bs_gamma_cash,bs_theta,alpha_bump,rho_bump,nu_bump
alpha_portfolio,31.3,8.919,1.014,60.5,-0.4831,1,1.359e-18,2.067e-16
rho_portfolio,-278.1,-108.4,-0.2388,-2.09,0.1551,1.236e-15,1,6.03e-16
nu_portfolio,-79.46,-35.63,-0.06781,-6.631,-0.004245,5.418e-16,-9.91e-18,1


In [77]:
# Bubble plot: weights by strike, bubble area = |weight| (scaled within each portfolio), hollow = short
fig = make_subplots(rows=1, cols=3, subplot_titles=list(weights.columns), horizontal_spacing=0.07)
colors = ["#1f77b4", "#d62728", "#2ca02c"]
for i, (ptf, color) in enumerate(zip(weights.columns, colors)):
    w = weights[ptf]
    fig.add_trace(
        go.Scatter(
            x=legs_risks["strike"], y=w, mode="markers+text", name=ptf,
            text=[f"{name}<br>{v:+.2f}" for name, v in w.items()], textposition="top center",
            marker=dict(
                size=w.abs(), sizemode="area", sizeref=2 * w.abs().max() / 60 ** 2, sizemin=4,
                color=[color if v >= 0 else "white" for v in w], line=dict(color=color, width=2),
            ),
            hovertemplate="K=%{x:.2f}<br>weight=%{y:.4g}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    pad = 0.45 * (w.max() - w.min())
    fig.update_yaxes(range=[w.min() - pad, w.max() + pad], row=1, col=i + 1)
fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.update_xaxes(title_text="strike", range=[legs_risks["strike"].min() - 3, legs_risks["strike"].max() + 3])
fig.update_yaxes(title_text="weight", row=1, col=1)
fig.update_layout(title="Subportfolios: option weights by strike", height=480, showlegend=False)
fig.show()

### Montecarlo

In [78]:
from scipy.special import ndtr

# Monte Carlo parameters
n_paths = 50_000                            # MC paths
n_steps = round(T * 252)                    # hedging steps (daily, held to maturity)
n_substeps = 10                             # simulation sub-steps per hedging step (limits the discretisation bias)
seed = 42                                   # random seed

# Realized parameters scanned around the implied ones (alpha, rho, nu of the Inputs section)
realized_grid = {
    "alpha": np.round(alpha + np.linspace(-0.04, 0.04, 9), 4),
    "rho": np.round(rho + np.linspace(-0.25, 0.25, 11), 4),
    "nu": np.round(nu + np.linspace(-0.5, 0.5, 11), 4),
}
implied = {"alpha": alpha, "rho": rho, "nu": nu}

# Same random numbers for every scenario, so differences between scenarios are not MC noise
rng = np.random.default_rng(seed)
Z_F = rng.standard_normal((n_steps * n_substeps, n_paths))
Z_perp = rng.standard_normal((n_steps * n_substeps, n_paths))


def simulate_sabr_paths(realized_alpha, realized_rho, realized_nu):
    """SABR paths dF = sigma * F^beta * dW, dsigma = nu * sigma * dZ, d<W, Z> = rho dt, sampled at the hedging dates.

    Returns (F, sigma) arrays of shape (n_steps + 1, n_paths).
    """
    dt = T / (n_steps * n_substeps)
    dW = Z_F * np.sqrt(dt)
    dZ = (realized_rho * Z_F + np.sqrt(1 - realized_rho ** 2) * Z_perp) * np.sqrt(dt)
    F = np.empty((n_steps + 1, n_paths))
    sigma = np.empty((n_steps + 1, n_paths))
    F[0], sigma[0] = f, s = np.full(n_paths, F0), np.full(n_paths, realized_alpha)
    for i in range(n_steps * n_substeps):
        local_vol = s * f ** (beta - 1)
        f = f * np.exp(local_vol * dW[i] - 0.5 * local_vol ** 2 * dt)
        s = s * np.exp(realized_nu * dZ[i] - 0.5 * realized_nu ** 2 * dt)
        if (i + 1) % n_substeps == 0:
            F[(i + 1) // n_substeps], sigma[(i + 1) // n_substeps] = f, s
    return F, sigma


def sabr_vol(F, K, T_left, alpha_t):
    """Hagan (2002) implied vol under the implied (pricing) beta, rho, nu, vectorised over paths."""
    fk_pow = (F * K) ** (0.5 * (1 - beta))
    log_fk = np.log(F / K)
    z = nu / alpha_t * fk_pow * log_fk
    x = np.log((np.sqrt(1 - 2 * rho * z + z ** 2) + z - rho) / (1 - rho))
    small = np.abs(z) < 1e-7
    z_over_x = np.where(small, 1.0, z / np.where(small, 1.0, x))
    denom = fk_pow * (1 + (1 - beta) ** 2 / 24 * log_fk ** 2 + (1 - beta) ** 4 / 1920 * log_fk ** 4)
    corr = 1 + ((1 - beta) ** 2 / 24 * alpha_t ** 2 / fk_pow ** 2
                + 0.25 * rho * beta * nu * alpha_t / fk_pow
                + (2 - 3 * rho ** 2) / 24 * nu ** 2) * T_left
    return alpha_t / denom * z_over_x * corr


def option_pv(F, K, T_left, alpha_t, is_call):
    """Black-76 price on the implied SABR smile, vectorised over paths."""
    sd = sabr_vol(F, K, T_left, alpha_t) * np.sqrt(T_left)
    d1 = np.log(F / K) / sd + 0.5 * sd
    call = F * ndtr(d1) - K * ndtr(d1 - sd)
    return np.exp(-r * T_left) * (call if is_call else call - (F - K))


def sabr_delta(F, K, T_left, alpha_t, is_call, rel_bump=1e-4):
    """SABR delta with the spot / vol correlation adjustment (Bartlett 2006):
    alpha moves by its expected shift dalpha = rho * nu / F^beta * dF when the forward is bumped."""
    h = F * rel_bump
    d_alpha = rho * nu / F ** beta * h
    up = option_pv(F + h, K, T_left, alpha_t + d_alpha, is_call)
    down = option_pv(F - h, K, T_left, alpha_t - d_alpha, is_call)
    return (up - down) / (2 * h)


def backtest(realized_alpha=alpha, realized_rho=rho, realized_nu=nu):
    """P&L per path of the 3 subportfolios, bought at the implied PV, held to maturity and delta hedged daily with the forward.

    Paths follow the realized parameters. Hedge deltas are computed with the implied rho / nu and an implied alpha
    trading at a constant ratio to the realized instantaneous vol: alpha_t = sigma_t * alpha / realized_alpha.
    Returns a DataFrame n_paths x (alpha_portfolio, rho_portfolio, nu_portfolio).
    """
    F, sigma = simulate_sabr_paths(realized_alpha, realized_rho, realized_nu)
    implied_alpha = sigma * alpha / realized_alpha
    times = np.linspace(0, T, n_steps + 1)
    legs_pnl = np.empty((n_paths, len(legs)))
    for j, (K, option_type) in enumerate(legs.values()):
        is_call = option_type == "call"
        payoff = np.maximum(F[-1] - K, 0.0) if is_call else np.maximum(K - F[-1], 0.0)
        hedge_pnl = sum(
            -sabr_delta(F[i], K, T - times[i], implied_alpha[i], is_call) * (F[i + 1] - F[i]) for i in range(n_steps)
        )
        legs_pnl[:, j] = payoff - legs_risks["PV"].iloc[j] + hedge_pnl
    return pd.DataFrame(legs_pnl @ weights.values, columns=weights.columns)

In [79]:
# Paths and P&L distribution when realized = implied: every portfolio is flat on average
F_paths, sigma_paths = simulate_sabr_paths(alpha, rho, nu)
base_pnl = backtest()
times_days = np.arange(n_steps + 1)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Forward paths (first 50)", "SABR vol paths (first 50)"])
for k in range(50):
    fig.add_trace(go.Scatter(x=times_days, y=F_paths[:, k], mode="lines", line=dict(width=0.8)), row=1, col=1)
    fig.add_trace(go.Scatter(x=times_days, y=sigma_paths[:, k], mode="lines", line=dict(width=0.8)), row=1, col=2)
fig.update_xaxes(title_text="business days")
fig.update_layout(height=380, showlegend=False)
fig.show()

display(pd.DataFrame({
    "mean": base_pnl.mean(), "std_error": base_pnl.sem(), "std": base_pnl.std(), "pct_paths_positive": (base_pnl > 0).mean() * 100
}))

,mean,std_error,std,pct_paths_positive
alpha_portfolio,-0.03837,0.02715,6.07,49.89
rho_portfolio,0.1929,0.285,63.73,47.98
nu_portfolio,0.03579,0.08379,18.74,48.08


In [80]:
# One realized parameter moved at a time, the two others kept at their implied value
bump_sizes = {"alpha": alpha_bump_size, "rho": rho_bump_size, "nu": nu_bump_size}
sweep = []
for param, values in realized_grid.items():
    for value in values:
        pnl = backtest(**{f"realized_{param}": value})
        for ptf in pnl.columns:
            sweep.append({"param": param, "realized": value, "portfolio": ptf,
                          "mean_pnl": pnl[ptf].mean(), "std_error": pnl[ptf].sem()})
sweep = pd.DataFrame(sweep)

# Mean P&L per unit bump of realized - implied: ~1 on the diagonal (own parameter), ~0 elsewhere
nearest = sweep[np.isclose(sweep["realized"] - sweep["param"].map(implied), sweep["param"].map(bump_sizes))]
display(nearest.pivot(index="portfolio", columns="param", values="mean_pnl").loc[weights.columns, list(implied)]
        .rename(columns=lambda p: f"realized {p} = implied + {bump_sizes[p]}"))

fig = make_subplots(
    rows=3, cols=3, shared_xaxes="columns", shared_yaxes="rows", horizontal_spacing=0.03, vertical_spacing=0.05,
    row_titles=list(weights.columns), column_titles=[f"realized {p} (implied = {implied[p]})" for p in implied],
)
for i, ptf in enumerate(weights.columns):
    for j, param in enumerate(implied):
        d = sweep[(sweep["portfolio"] == ptf) & (sweep["param"] == param)]
        fig.add_trace(
            go.Bar(
                x=d["realized"], y=d["mean_pnl"], marker_color=np.where(d["mean_pnl"] >= 0, "#2ca02c", "#d62728"),
                error_y=dict(type="data", array=2 * d["std_error"], color="grey", thickness=1),
                hovertemplate="realized " + param + "=%{x}<br>mean P&L=%{y:.3f}<extra></extra>",
            ),
            row=i + 1, col=j + 1,
        )
        fig.add_vline(x=implied[param], line_dash="dot", line_color="black", row=i + 1, col=j + 1)
fig.update_yaxes(title_text="mean P&L", col=1)
fig.update_layout(
    title="Mean delta hedged P&L vs realized parameter (green: positive, red: negative, dotted line: implied, bars: 2 std errors)",
    height=800, showlegend=False, bargap=0.15,
)
fig.show()

param,realized alpha = implied + 0.01,realized rho = implied + 0.1,realized nu = implied + 0.5
alpha_portfolio,0.9581,-0.03741,-0.05013
rho_portfolio,0.1895,1.156,0.3362
nu_portfolio,0.03113,0.02334,1.021


In [81]:
# Realized rho and nu moved together (realized alpha = implied): sign of the mean P&L of each portfolio
heat = {ptf: pd.DataFrame(index=realized_grid["nu"], columns=realized_grid["rho"], dtype=float) for ptf in weights.columns}
for realized_rho in realized_grid["rho"]:
    for realized_nu in realized_grid["nu"]:
        pnl = backtest(realized_rho=realized_rho, realized_nu=realized_nu).mean()
        for ptf in weights.columns:
            heat[ptf].loc[realized_nu, realized_rho] = pnl[ptf]

fig = make_subplots(rows=1, cols=3, subplot_titles=list(weights.columns), shared_yaxes=True, horizontal_spacing=0.04)
for i, ptf in enumerate(weights.columns):
    z = heat[ptf]
    zmax = np.abs(z.values).max()
    fig.add_trace(
        go.Heatmap(
            x=z.columns, y=z.index, z=z.values, colorscale="RdYlGn", zmid=0, zmin=-zmax, zmax=zmax,
            text=z.round(1).values, texttemplate="%{text}", textfont=dict(size=9), showscale=False,
            hovertemplate="realized rho=%{x}<br>realized nu=%{y}<br>mean P&L=%{z:.3f}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    fig.add_trace(
        go.Scatter(x=[rho], y=[nu], mode="markers", marker=dict(symbol="x", size=12, color="black"), name="implied"),
        row=1, col=i + 1,
    )
fig.update_xaxes(title_text="realized rho")
fig.update_yaxes(title_text="realized nu", col=1)
fig.update_layout(
    title="Mean delta hedged P&L by realized rho / nu (green: positive, red: negative, x: implied)",
    height=520, showlegend=False,
)
fig.show()

### Montecarlo rehedging

In [82]:
from scipy.special import ndtri

# Rehedging parameters
rehedge_n_paths = 20_000                    # paths used in this section (first paths of the Montecarlo section)
last_rehedge_days_to_expiry = 5             # no option trade closer to expiry than this: the bumps of the 3 options collapse
keep_own_exposure = False                   # False: desired exposure reset to 1 every day; True: trades add no desired exposure (unstable: it then drifts and compounds)
bump_names = ["alpha", "rho", "nu"]


def smile_sd(F, K, T_left, alpha_t, d_rho=0.0, d_nu=0.0):
    """Implied SABR vol * sqrt(T_left) (Hagan 2002, optionally with rho / nu bumped), broadcast over options x paths."""
    rho_, nu_ = rho + d_rho, nu + d_nu
    fk_pow = (F * K) ** (0.5 * (1 - beta))
    log_fk = np.log(F / K)
    z = nu_ / alpha_t * fk_pow * log_fk
    x = np.log((np.sqrt(1 - 2 * rho_ * z + z ** 2) + z - rho_) / (1 - rho_))
    small = np.abs(z) < 1e-7
    z_over_x = np.where(small, 1.0, z / np.where(small, 1.0, x))
    denom = fk_pow * (1 + (1 - beta) ** 2 / 24 * log_fk ** 2 + (1 - beta) ** 4 / 1920 * log_fk ** 4)
    corr = 1 + ((1 - beta) ** 2 / 24 * alpha_t ** 2 / fk_pow ** 2
                + 0.25 * rho_ * beta * nu_ * alpha_t / fk_pow
                + (2 - 3 * rho_ ** 2) / 24 * nu_ ** 2) * T_left
    return alpha_t / denom * z_over_x * corr * np.sqrt(T_left)


def smile_pv(F, K, T_left, alpha_t, is_call, d_rho=0.0, d_nu=0.0):
    """Black-76 price on the implied SABR smile, broadcast over options x paths."""
    sd = smile_sd(F, K, T_left, alpha_t, d_rho, d_nu)
    d1 = np.log(F / K) / sd + 0.5 * sd
    call = F * ndtr(d1) - K * ndtr(d1 - sd)
    return np.exp(-r * T_left) * np.where(is_call, call, call - (F - K))


def option_risks(F, K, T_left, alpha_t, is_call, rel_bump=1e-4):
    """PV, SABR delta (Bartlett) and (alpha, rho, nu) bumps of each option on each path. bumps: (3, n_options, n_paths)."""
    pv = smile_pv(F, K, T_left, alpha_t, is_call)
    bumps = np.stack([
        smile_pv(F, K, T_left, alpha_t + alpha_bump_size, is_call) - pv,
        smile_pv(F, K, T_left, alpha_t, is_call, d_rho=rho_bump_size) - pv,
        smile_pv(F, K, T_left, alpha_t, is_call, d_nu=nu_bump_size) - pv,
    ])
    h = F * rel_bump
    d_alpha = rho * nu / F ** beta * h
    delta = (smile_pv(F + h, K, T_left, alpha_t + d_alpha, is_call)
             - smile_pv(F - h, K, T_left, alpha_t - d_alpha, is_call)) / (2 * h)
    return pv, delta, bumps


def trading_strikes(F, T_left, alpha_t, n_iter=10):
    """Strikes of the trading portfolio on each path: 25d put, ATM (= forward), 25d call on the current implied smile.

    Fixed point on K = F * exp(-N^-1(call delta) * sd(K) + sd(K)^2 / 2), sd(K) = SABR vol(K) * sqrt(T_left). Shape (3, n_paths).
    """
    strikes = [F]
    for call_delta in (0.75, 0.25):          # 25d put = 75d call
        K = F
        for _ in range(n_iter):
            sd = smile_sd(F, K, T_left, alpha_t)
            K = F * np.exp(-ndtri(call_delta) * sd + 0.5 * sd ** 2)
        strikes.append(K)
    return np.stack([strikes[1], strikes[0], strikes[2]])


def backtest_rehedged(realized_alpha=alpha, realized_rho=rho, realized_nu=nu, rehedge=True):
    """Same backtest as `backtest`, but every day each portfolio buys 3 options (current 25d put, ATM, 25d call, same expiry)
    sized so that its exposure to the two undesired bumps is back to 0 (3x3 solve per path). rehedge=False: static portfolios.

    Returns (pnl, exposures, gross_traded):
        pnl: DataFrame n_paths x portfolio
        exposures: array (n_steps, bump, portfolio), mean over paths of |exposure| of the book before the daily trade
        gross_traded: DataFrame step x portfolio, mean over paths of sum |weights| traded
    """
    F, sigma = simulate_sabr_paths(realized_alpha, realized_rho, realized_nu)
    F, sigma = F[:, :rehedge_n_paths], sigma[:, :rehedge_n_paths]
    implied_alpha = sigma * alpha / realized_alpha
    n = F.shape[1]
    n_ptf = weights.shape[1]
    is_call_legs = np.array([False, True, True])[:, None]            # 25d put, ATM (call), 25d call

    K = np.empty((0, n))                                              # strikes held, (n_options, n_paths)
    is_call = np.empty((0, 1), dtype=bool)
    W = np.empty((0, n, n_ptf))                                       # weights held, (n_options, n_paths, portfolio)
    pv_prev = np.empty((0, n))
    pnl = np.zeros((n, n_ptf))
    exposures = np.zeros((n_steps, 3, n_ptf))
    gross_traded = np.zeros((n_steps, n_ptf))

    for i in range(n_steps):
        T_left = T - i / 252
        pv, delta, bumps = option_risks(F[i], K, T_left, implied_alpha[i], is_call)
        pnl += np.einsum("kn,knp->np", pv - pv_prev, W)              # option P&L of the step ending today
        exposure = np.einsum("bkn,knp->nbp", bumps, W)               # (n_paths, bump, portfolio) before today's trade
        exposures[i] = np.abs(exposure).mean(axis=0)

        if i == 0 or (rehedge and n_steps - i >= last_rehedge_days_to_expiry):
            if i == 0:
                K_new = np.repeat(legs_risks["strike"].values[:, None], n, axis=1)
            else:
                K_new = trading_strikes(F[i], T_left, implied_alpha[i])
            pv_new, delta_new, bumps_new = option_risks(F[i], K_new, T_left, implied_alpha[i], is_call_legs)
            target = np.where(np.eye(3, dtype=bool), exposure if (keep_own_exposure and i > 0) else 1.0, 0.0)
            # B[n, bump, option] @ x[n, option, portfolio] = target - exposure
            x = np.linalg.solve(bumps_new.transpose(2, 0, 1), target - exposure)
            K, is_call = np.vstack([K, K_new]), np.vstack([is_call, is_call_legs])
            W = np.concatenate([W, x.transpose(1, 0, 2)])
            pv, delta = np.vstack([pv, pv_new]), np.vstack([delta, delta_new])
            gross_traded[i] = np.abs(x).sum(axis=1).mean(axis=0)

        pnl -= np.einsum("kn,knp->np", delta, W) * (F[i + 1] - F[i])[:, None]   # forward hedge over the next step
        pv_prev = pv

    payoff = np.where(is_call, np.maximum(F[-1] - K, 0.0), np.maximum(K - F[-1], 0.0))
    pnl += np.einsum("kn,knp->np", payoff - pv_prev, W)
    return (
        pd.DataFrame(pnl, columns=weights.columns),
        exposures,
        pd.DataFrame(gross_traded, columns=weights.columns).rename_axis("step"),
    )

In [83]:
# Realized = implied: static portfolios vs portfolios rehedged every day with the trading portfolio
static_pnl, static_exposures, _ = backtest_rehedged(rehedge=False)
rehedged_pnl, rehedged_exposures, gross_traded = backtest_rehedged()

display(pd.concat({
    "static": pd.DataFrame({"mean": static_pnl.mean(), "std_error": static_pnl.sem(), "std": static_pnl.std()}),
    "rehedged": pd.DataFrame({"mean": rehedged_pnl.mean(), "std_error": rehedged_pnl.sem(), "std": rehedged_pnl.std()}),
}, axis=1))

# Exposure of each book before the daily trade (mean over paths of the absolute value): the rehedged book only carries
# one day of drift on the two undesired bumps, the static book drifts freely
fig = make_subplots(rows=1, cols=3, subplot_titles=list(weights.columns), horizontal_spacing=0.06)
bump_colors = {"alpha": "#1f77b4", "rho": "#d62728", "nu": "#2ca02c"}
days = np.arange(n_steps)
for j, ptf in enumerate(weights.columns):
    for b, bump in enumerate(bump_names):
        own = ptf.startswith(bump)
        for label, exposures, dash in (("rehedged", rehedged_exposures, "solid"), ("static", static_exposures, "dash")):
            fig.add_trace(
                go.Scatter(
                    x=days[1:], y=exposures[1:, b, j], mode="lines", name=f"{bump}_bump, {label}",
                    legendgroup=f"{bump}{label}", showlegend=j == 0,
                    line=dict(color=bump_colors[bump], dash=dash, width=3 if own else 1.5),
                ),
                row=1, col=j + 1,
            )
    fig.add_vline(x=n_steps - last_rehedge_days_to_expiry, line_dash="dot", line_color="grey", row=1, col=j + 1)
fig.update_xaxes(title_text="business days")
fig.update_yaxes(title_text="mean |exposure| before the daily trade", col=1)
fig.update_layout(
    title="Bump exposures through time (thick: desired exposure, dashed: static portfolio, dotted line: last rehedge)", height=450,
)
fig.show()

# Size of the daily trades: sum of |weights| of the 3 options traded, mean over paths (day 0 = initial portfolio)
fig = go.Figure([go.Bar(x=gross_traded.index, y=gross_traded[ptf], name=ptf) for ptf in weights.columns])
fig.update_layout(title="Options traded per day (sum of |weights|, mean over paths)", xaxis_title="business days",
                  yaxis_title="gross weight", yaxis_type="log", barmode="group", height=400)
fig.show()

static                 rehedged                
                    mean std_error   std     mean std_error   std
alpha_portfolio -0.05724   0.04295 6.074 -0.03922   0.08291 11.72
rho_portfolio     0.6601    0.4517 63.88   0.2646     2.046 289.4
nu_portfolio      0.1891     0.133  18.8  0.02073    0.5864 82.93

In [84]:
# Same scan as the Montecarlo section: one realized parameter moved at a time
rehedged_sweep = []
for param, values in realized_grid.items():
    for value in values:
        pnl = backtest_rehedged(**{f"realized_{param}": value})[0]
        for ptf in pnl.columns:
            rehedged_sweep.append({"param": param, "realized": value, "portfolio": ptf,
                                   "mean_pnl": pnl[ptf].mean(), "std_error": pnl[ptf].sem()})
rehedged_sweep = pd.DataFrame(rehedged_sweep)

# Mean P&L when one realized parameter is one bump above implied: rehedged vs static
def one_bump_table(df):
    d = df[np.isclose(df["realized"] - df["param"].map(implied), df["param"].map(bump_sizes))]
    return (d.pivot(index="portfolio", columns="param", values="mean_pnl").loc[weights.columns, list(implied)]
            .rename(columns=lambda p: f"realized {p} = implied + {bump_sizes[p]}"))

display(pd.concat({"static": one_bump_table(sweep), "rehedged": one_bump_table(rehedged_sweep)}))

fig = make_subplots(
    rows=3, cols=3, shared_xaxes="columns", shared_yaxes="rows", horizontal_spacing=0.03, vertical_spacing=0.05,
    row_titles=list(weights.columns), column_titles=[f"realized {p} (implied = {implied[p]})" for p in implied],
)
for i, ptf in enumerate(weights.columns):
    for j, param in enumerate(implied):
        d = rehedged_sweep[(rehedged_sweep["portfolio"] == ptf) & (rehedged_sweep["param"] == param)]
        s = sweep[(sweep["portfolio"] == ptf) & (sweep["param"] == param)]
        fig.add_trace(
            go.Bar(
                x=d["realized"], y=d["mean_pnl"], marker_color=np.where(d["mean_pnl"] >= 0, "#2ca02c", "#d62728"),
                error_y=dict(type="data", array=2 * d["std_error"], color="grey", thickness=1), name="rehedged",
                hovertemplate="realized " + param + "=%{x}<br>rehedged mean P&L=%{y:.3f}<extra></extra>",
            ),
            row=i + 1, col=j + 1,
        )
        fig.add_trace(
            go.Scatter(x=s["realized"], y=s["mean_pnl"], mode="lines+markers", line=dict(color="black", width=1.5),
                       marker=dict(size=5), name="static",
                       hovertemplate="realized " + param + "=%{x}<br>static mean P&L=%{y:.3f}<extra></extra>"),
            row=i + 1, col=j + 1,
        )
        fig.add_vline(x=implied[param], line_dash="dot", line_color="black", row=i + 1, col=j + 1)
fig.update_yaxes(title_text="mean P&L", col=1)
fig.update_layout(
    title="Rehedged portfolios: mean delta hedged P&L vs realized parameter (bars: rehedged, black line: static, dotted line: implied)",
    height=800, showlegend=False, bargap=0.15,
)
fig.show()

param                     realized alpha = implied + 0.01  \
static   alpha_portfolio                           0.9581   
         rho_portfolio                             0.1895   
         nu_portfolio                             0.03113   
rehedged alpha_portfolio                            2.378   
         rho_portfolio                            -0.7441   
         nu_portfolio                             -0.3834   

param                     realized rho = implied + 0.1  \
static   alpha_portfolio                      -0.03741   
         rho_portfolio                           1.156   
         nu_portfolio                          0.02334   
rehedged alpha_portfolio                      -0.09459   
         rho_portfolio                            4.42   
         nu_portfolio                           0.1871   

param                     realized nu = implied + 0.5  
static   alpha_portfolio                     -0.05013  
         rho_portfolio                         0.3362  
         nu_portfolio                           1.021  
rehedged alpha_portfolio                      -0.1424  
         rho_portfolio                          1.933  
         nu_portfolio                            3.97